# 🤖 Agentic RAG with Follow-Up Retrieval

## What We're Building

A RAG agent that **decides on its own** whether it has enough information
to answer, or if it needs to retrieve more documents. Unlike fixed pipelines,
the agent can:

1. **Retrieve** documents for a question
2. **Evaluate** if the retrieved context is sufficient
3. **Reformulate** and retrieve again if needed
4. **Answer** once it has enough evidence

## Why Agentic RAG?

Standard RAG retrieves once and answers. But some questions need:
- A second retrieval with a refined query
- Context from a different topic area
- Clarification before answering

## The Agent Loop

```
User Question
     ↓
  Retrieve (first pass)
     ↓
  Evaluate: Enough to answer?  ──No──→  Reformulate query → Retrieve again
     ↓ Yes                                       ↓
  Generate Answer  ←─────────────────────────────┘
```

## Stack
- **LangGraph** — stateful agent graph
- **LangChain** — retrieval + LLM
- **ChromaDB** — vector store
- **Ollama** — local LLM

## Step 1 — Imports & Knowledge Base

In [ ]:
# !pip install langgraph langchain langchain-ollama chromadb -q

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.text_splitter import RecursiveCharacterTextSplitter

print("All imports successful!")

In [ ]:
# Broad knowledge base covering multiple interconnected topics
knowledge = """
# TechStart Company Wiki

## Product: DataPipeline
DataPipeline is our flagship ETL product. It extracts data from 50+ sources,
transforms it using SQL or Python, and loads into data warehouses. It supports
Snowflake, BigQuery, Redshift, and Databricks as destinations.

## Product: QueryEngine
QueryEngine is our analytics product. It provides a SQL interface over data
warehouses with caching, query optimization, and access control. It integrates
tightly with DataPipeline for end-to-end data flows.

## Architecture
Both products run on Kubernetes (EKS). DataPipeline workers are horizontally
scalable — each worker processes one pipeline run. QueryEngine uses a query
planner that distributes work across multiple executor pods.

## DataPipeline Pricing
- Starter: $99/mo — 10 pipelines, hourly scheduling
- Growth: $499/mo — 100 pipelines, 5-min scheduling, Slack alerts
- Enterprise: Custom — unlimited, real-time, SLA, dedicated support

## QueryEngine Pricing
- Free tier: 100 queries/month, 10GB scanned
- Pro: $199/mo — unlimited queries, 1TB/month, caching
- Enterprise: Custom — multi-tenant, SSO, audit logging

## Security
SOC 2 Type II certified. All data encrypted at rest (AES-256) and in transit
(TLS 1.3). Row-level security in QueryEngine. API keys with scoped permissions.
SAML SSO available for Enterprise tier. IP allowlisting supported.

## DataPipeline Connectors
Sources: PostgreSQL, MySQL, MongoDB, Salesforce, HubSpot, Stripe, Shopify,
Google Sheets, REST APIs, S3 files (CSV, Parquet, JSON).
Destinations: Snowflake, BigQuery, Redshift, Databricks, S3, PostgreSQL.

## Error Handling
DataPipeline retries failed tasks 3 times with exponential backoff.
After 3 failures, the pipeline is marked as failed and alerts are sent.
Partial runs can be resumed from the last successful checkpoint.
QueryEngine returns detailed error messages with query plan diagnostics.

## Performance Benchmarks
DataPipeline: 100K rows/second for PostgreSQL sources. 500K rows/second
for Parquet files on S3. Latency depends on source and transformation complexity.
QueryEngine: P50 = 200ms, P95 = 2s, P99 = 8s for cached queries.
Uncached complex queries can take 10-60 seconds depending on data volume.

## Team & Support
Engineering: 45 engineers across 6 teams (Platform, DataPipeline Core,
Connectors, QueryEngine, Security, DevEx).
Support: Starter gets email support (48h SLA). Growth gets Slack + email (8h).
Enterprise gets dedicated Slack channel + phone (1h for P0).

## Roadmap Q1 2025
- DataPipeline: Apache Iceberg destination support, dbt integration
- QueryEngine: Semantic layer, AI-powered query suggestions
- Platform: Usage-based billing, improved onboarding wizard
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50, separators=["\n## ", "\n\n", "\n"])
docs = splitter.create_documents([knowledge])

embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(docs, embeddings, collection_name="agentic_rag")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

print(f"Indexed {len(docs)} chunks")

## Step 2 — Define the Agent State

LangGraph uses a **typed state dict** that flows through the graph.
Each node reads from and writes to this state.

In [ ]:
class AgentState(TypedDict):
    question: str                  # Original user question
    search_query: str              # Current query for retrieval (may be rewritten)
    retrieved_docs: list[str]      # All retrieved document contents
    is_sufficient: bool            # Does the agent have enough context?
    answer: str                    # Final answer
    iteration: int                 # How many retrieval rounds
    reasoning: str                 # Agent's reasoning about sufficiency

print("AgentState defined")

## Step 3 — Define Agent Nodes

Each node is a function that takes the state and returns updates.

In [ ]:
# Node 1: Retrieve documents
def retrieve(state: AgentState) -> dict:
    """Retrieve documents for the current search query."""
    query = state["search_query"] or state["question"]
    docs = retriever.invoke(query)
    new_docs = [d.page_content for d in docs]

    # Merge with existing docs (deduplicate)
    existing = state.get("retrieved_docs", []) or []
    all_docs = list(existing)
    for d in new_docs:
        if d not in all_docs:
            all_docs.append(d)

    iteration = (state.get("iteration") or 0) + 1
    print(f"  🔍 Retrieve (round {iteration}): query='{query[:50]}...' → {len(new_docs)} new docs, {len(all_docs)} total")

    return {"retrieved_docs": all_docs, "iteration": iteration}


# Node 2: Evaluate if we have enough context
eval_prompt = ChatPromptTemplate.from_template(
    """Given the question and the retrieved context, decide if there is
ENOUGH information to fully answer the question.

Question: {question}

Retrieved Context:
{context}

Respond with EXACTLY one of:
- SUFFICIENT: if the context contains enough information to answer
- INSUFFICIENT: <what's missing> — if key information is missing

Decision:"""
)


def evaluate(state: AgentState) -> dict:
    """Evaluate if retrieved context is sufficient to answer."""
    context = "\n\n---\n\n".join(state["retrieved_docs"][:6])
    chain = eval_prompt | llm | StrOutputParser()
    result = chain.invoke({"question": state["question"], "context": context})

    is_sufficient = "SUFFICIENT" in result.upper().split("\n")[0] and "INSUFFICIENT" not in result.upper().split("\n")[0]

    # Cap iterations to prevent infinite loops
    if state.get("iteration", 0) >= 3:
        is_sufficient = True
        result = "Max iterations reached — answering with available context"

    print(f"  🤔 Evaluate: {'✅ Sufficient' if is_sufficient else '❌ Insufficient'} — {result[:80]}")

    return {"is_sufficient": is_sufficient, "reasoning": result}


# Node 3: Reformulate query for follow-up retrieval
reformulate_prompt = ChatPromptTemplate.from_template(
    """The current retrieved context doesn't fully answer the question.
Generate a NEW search query to find the missing information.

Question: {question}
What's missing: {reasoning}

Already retrieved:
{context}

Output ONLY the new search query (1 sentence):"""
)


def reformulate(state: AgentState) -> dict:
    """Generate a new search query targeting missing information."""
    context = "\n".join(state["retrieved_docs"][:3])
    chain = reformulate_prompt | llm | StrOutputParser()
    new_query = chain.invoke({
        "question": state["question"],
        "reasoning": state["reasoning"],
        "context": context,
    })
    print(f"  🔄 Reformulate: '{new_query[:60]}...'")
    return {"search_query": new_query}


# Node 4: Generate final answer
answer_prompt = ChatPromptTemplate.from_template(
    """Answer the question using the provided context. Be comprehensive.
Cite specific facts from the context.

Context:
{context}

Question: {question}

Answer:"""
)


def generate_answer(state: AgentState) -> dict:
    """Generate the final answer from accumulated context."""
    context = "\n\n---\n\n".join(state["retrieved_docs"][:6])
    chain = answer_prompt | llm | StrOutputParser()
    answer = chain.invoke({"question": state["question"], "context": context})
    print(f"  ✅ Answer generated ({len(answer)} chars, used {len(state['retrieved_docs'])} docs)")
    return {"answer": answer}


print("All agent nodes defined")

## Step 4 — Build the Agent Graph

Connect nodes with conditional edges — the key is the
`evaluate` → `reformulate` or `answer` routing.

In [ ]:
def route_after_eval(state: AgentState) -> str:
    """Route based on evaluation result."""
    if state["is_sufficient"]:
        return "generate_answer"
    else:
        return "reformulate"


# Build the graph
graph = StateGraph(AgentState)

# Add nodes
graph.add_node("retrieve", retrieve)
graph.add_node("evaluate", evaluate)
graph.add_node("reformulate", reformulate)
graph.add_node("generate_answer", generate_answer)

# Add edges
graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "evaluate")
graph.add_conditional_edges("evaluate", route_after_eval)
graph.add_edge("reformulate", "retrieve")
graph.add_edge("generate_answer", END)

# Compile
agent = graph.compile()

print("Agent graph compiled!")
print("\nGraph structure:")
print("  retrieve → evaluate → [sufficient?]")
print("                            ├─ Yes → generate_answer → END")
print("                            └─ No  → reformulate → retrieve (loop)")

## Step 5 — Run the Agent

In [ ]:
def ask_agent(question: str) -> None:
    """Run the agentic RAG pipeline."""
    print(f"\n❓ {question}")
    print("=" * 60)

    initial_state = {
        "question": question,
        "search_query": question,
        "retrieved_docs": [],
        "is_sufficient": False,
        "answer": "",
        "iteration": 0,
        "reasoning": "",
    }

    result = agent.invoke(initial_state)

    print(f"\n📝 Final Answer (after {result['iteration']} retrieval rounds):")
    print(result["answer"])
    print("=" * 60)

In [ ]:
# Simple question — should answer in 1 round
ask_agent("What databases does DataPipeline support as sources?")

In [ ]:
# Cross-topic question — may need follow-up retrieval
ask_agent("Compare the pricing and performance of DataPipeline vs QueryEngine")

In [ ]:
# Question spanning security + roadmap
ask_agent("What security certifications does the platform have, and what's planned for Q1 2025?")

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **LangGraph StateGraph** | Manages agent state across nodes |
| **Conditional edges** | Routes flow based on evaluation |
| **Sufficiency check** | LLM decides if context is enough |
| **Query reformulation** | Targets missing information specifically |
| **Iteration cap** | Prevents infinite retrieval loops |
| **Context accumulation** | Follow-up rounds ADD to existing docs |

## 🔧 Customization Ideas

- **Tool-based agent**: Give the agent tools (web search, DB query, calculator)
- **Human-in-the-loop**: Ask the user for clarification at the evaluate step
- **Parallel retrieval**: Search multiple sources simultaneously
- **Self-reflection**: After answering, verify the answer against context